In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
import numpy as np

def main():
    print("1. Loading data...")
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test_input.csv')

    label_encoder = LabelEncoder()
    train_df['encoded_label'] = label_encoder.fit_transform(train_df['label'])
    num_labels = len(label_encoder.classes_)

    train_dataset = Dataset.from_pandas(train_df[['sentence', 'encoded_label']].rename(columns={'encoded_label': 'label'}))
    test_dataset = Dataset.from_pandas(test_df[['sentence']])

    print("2. Loading Tokenizer and Model...")
    model_name = "roberta-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    print("3. Tokenizing data...")
    def tokenize_function(examples):
        return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    print("4. Setting up Training Arguments...")
    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_steps=10,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
    )

    print("5. Fine-tuning the model (This will take a few minutes on a GPU)...")
    trainer.train()

    print("6. Making predictions on the test set...")
    predictions_output = trainer.predict(tokenized_test)

    predicted_classes = np.argmax(predictions_output.predictions, axis=1)

    print("7. Saving predictions.csv for CodaBench...")
    final_labels = label_encoder.inverse_transform(predicted_classes)

    submission_df = test_df.copy()
    submission_df['label'] = final_labels

    submission_df = submission_df[['sentence', 'label']]
    submission_df.to_csv('predictions.csv', index=False)

    print("Done! Your predictions.csv is ready.")

if __name__ == "__main__":
    main()

1. Loading data...
2. Loading Tokenizer and Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
10,1.936566
20,1.386906
30,1.052462
40,1.179764
50,0.944858
60,0.975202
70,1.134182
80,1.021047
90,0.935672
100,1.004403


6. Making predictions on the test set...


7. Saving predictions.csv for CodaBench...
Done! Your predictions.csv is ready.
